# MAA — S9c: SAFER, ARC dan pembanding (63 unit)

Varian baru: A4+U5, A4+C1, A4+C2, A4+C1+C2, A4+OCC, A4+ASM, A4+MIX. Unit S9 yang sudah selesai dilewati.
Ringkasan dibandingkan dengan A0 dan dengan A4.

Unit S9 yang sudah selesai dilewati otomatis; ringkasan mencakup semua varian S9.


**SZ (utama):** latih di subset non-kursi roda (tanpa subjek yang ada di tes kursi roda) → uji di split subject-test kursi roda.
Sebagian subjek tes non-kursi roda ikut diuji untuk cek bahwa augmentasi tidak merugikan mereka.
**SO (pembanding atas):** latih dan uji di subset kursi roda sendiri.

ST-GCN, ST-GCN++, CTR-GCN × A0, A4, U5, U+U5, U1+U5, A4+U1+U5 × 3 seed (SZ, 54 unit) + A0 × 3 seed (SO, 9 unit).
Kelas = aksi yang ada di kedua subset dengan ≥10 segmen di tes kursi roda. Unit evaluasi = segmen aksi per kamera, macro-F1.

**Cara pakai:** *Runtime → T4 GPU* → **Run all** di beberapa sesi, sama seperti S1/S3. Sesi pertama mengonversi SAFER sekali
(±15–30 menit, sesi lain menunggu otomatis). Butuh folder SAFER (`MyDrive/CARR/SAFER` atau `MyDrive/SAFER`) dari notebook unduh.
Hasil: `MyDrive/CARR/maa/results/S9_*.csv`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA   = '/content/drive/MyDrive/CARR'
MODELS = ['ST-GCN', 'ST-GCN++', 'CTR-GCN']
AUGS_SZ, AUGS_SO = ['A0', 'A4', 'A4+U5', 'A4+C1+C2', 'A4+C1', 'A4+C2', 'A4+OCC', 'A4+ASM', 'A4+MIX'], ['A0']
SEEDS  = [0, 1, 2]


In [ ]:
import os
os.environ['CARR_REPO'] = '/content/CARR'
os.environ['CARR_LOCK_WAIT'] = '30'
!git clone -q https://github.com/januaradip/CARR.git /content/CARR 2>/dev/null; git -C /content/CARR fetch -q
!git -C /content/CARR checkout -q 1193346
import torch
if not torch.cuda.is_available():
    raise SystemExit('Sesi ini TIDAK memakai GPU. Runtime -> Change runtime type -> T4 GPU, lalu Run all lagi.')
print('GPU OK:', torch.cuda.get_device_name(0))

In [ ]:
%%writefile /content/maa.py
"""
MAA: Mobility-Aid-Aware skeleton augmentation for activity recognition on CARR.

Experiment S1 (this version): baselines without the proposed augmentation (aug 'A0') for four backbones
(ST-GCN, ST-GCN++, CTR-GCN, PoseC3D) on the nine CARR protocols (CS1-4, CL1-4, CG) and several seeds.

Builds on the public CARR benchmark code (github.com/januaradip/CARR):
  * frame index, protocol splits, run locks, trainer with checkpoint/resume  -> carr.common
  * 17 COCO keypoints (YOLOv8m-Pose), box-normalised sequences, ST-GCN / CTR-GCN -> carr.skeleton

Layout of the output (WORK):
  WORK/cache/frames.pkl, WORK/cache/seq_box.pkl         shared caches
  WORK/maa/runs/<exp>/<aug>/<model>/<split>/s<seed>/    checkpoints, DONE.json, video_probs.npz
  WORK/maa/results/<exp>_runs.csv, <exp>_summary.csv    collected results

Every unit (exp, aug, model, split, seed) is claimed with a lock file, so several Colab sessions (also from
different Google accounts sharing the same Drive folder) can work on the same experiment in parallel.
"""
import os, sys, glob, json, math, time, random, shutil, collections
import numpy as np

CARR_REPO = os.environ.get('CARR_REPO', '/content/CARR')
if CARR_REPO not in sys.path:
    sys.path.insert(0, CARR_REPO)
from carr import common as C                   # noqa: E402
from carr import skeleton as SK                # noqa: E402

MODELS = ['ST-GCN', 'ST-GCN++', 'CTR-GCN', 'PoseC3D']
SPLITS = ['CS1', 'CS2', 'CS3', 'CS4', 'CL1', 'CL2', 'CL3', 'CL4', 'CG']

CFG = dict(
    WIN=64, STRIDE=32,                         # windows as in the CARR benchmark
    EPOCHS=60, PATIENCE=12, WARMUP=3, WD=1e-4, WORKERS=2,
    BATCH={'ST-GCN': 64, 'ST-GCN++': 64, 'CTR-GCN': 64, 'PoseC3D': 32},
    ACCUM={'ST-GCN': 1, 'ST-GCN++': 1, 'CTR-GCN': 1, 'PoseC3D': 2},     # PoseC3D: effective batch 64
    LR={'ST-GCN': 1e-3, 'ST-GCN++': 1e-3, 'CTR-GCN': 1e-3, 'PoseC3D': 1e-3},
    HM_SIZE=56, HM_SIGMA=0.6, HM_RANGE=0.75, HM_FRAMES=32,              # PoseC3D heatmap volumes
)


# =============================================================================== data
def paths(data, work):
    return C.Paths(data, work)


def load_seq(P, F, pose_dir=None):
    """Box-normalised keypoint sequences {video: (T,17,3)}, cached once in WORK/cache/seq_box.pkl.
    pose_dir: folder with <video>.npz from the CARR benchmark (default WORK/cache/pose)."""
    cache = f'{P.cache}/seq_box.pkl'
    if os.path.exists(cache):
        obj = C.load_pickle(cache)
        if obj.get('sig') == P.signature() and len(obj['seq']) == len(F['videos']):
            return obj['seq']
    pose_dir = pose_dir or f'{P.cache}/pose'
    missing = [v for v in F['videos'] if not os.path.exists(f'{pose_dir}/{v}.npz')]
    if missing:
        raise FileNotFoundError(
            f'{len(missing)} of {len(F["videos"])} pose files missing in {pose_dir} (e.g. {missing[:3]}). '
            f'Set POSE_DIR to the pose cache of the CARR benchmark (CARR_work/cache/pose) or run the '
            f'pose-extraction cell.')
    C.log('Building box-normalised sequences (once)...')
    # carr.skeleton.load_sequences reads <cache>/pose/<video>.npz
    if os.path.basename(pose_dir.rstrip('/')) != 'pose':
        raise ValueError('POSE_DIR must be a folder named "pose" (e.g. .../CARR_work/cache/pose)')
    q = type('Q', (), {})()
    q.cache = os.path.dirname(pose_dir.rstrip('/'))
    seq = SK.load_sequences(q, F)
    C.save_pickle(dict(sig=P.signature(), seq=seq), cache)
    C.log(f'{len(seq)} sequences cached in {cache}')
    return seq


def make_windows(videos, seq):
    return SK.make_windows(videos, seq, None)


# ------------------------------------------------------------------ augmentations
def base_aug(x):
    """Augmentation of the CARR benchmark (kept for every variant): flip, rotation +-10 deg, scale 0.9-1.1."""
    if random.random() < 0.5:
        x = x[:, SK.FLIP]
        x[:, :, 0] *= -1
    a = math.radians(random.uniform(-10, 10))
    sc = random.uniform(0.9, 1.1)
    R = np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]], np.float32) * sc
    x[:, :, :2] = x[:, :, :2] @ R.T
    return x


try:
    import augs as _AG                         # generic (A1-A4) and proposed (U, U1-U4, U5) augmentations
    AUGS = dict(_AG.AUGS)
except ImportError:
    _AG, AUGS = None, {'A0': lambda x: x}


FLAGS = ('U5', 'C1', 'C2', 'ASM', 'MIX')       # representation / dataset-level parts of a variant name
_FILLED = {}                                    # C2-filled sequences, computed once per process and sequence dict


def parse_aug(name):
    """'A4+C1+C2' -> ('A4', {'C1', 'C2'}); 'U+U5' -> ('U', {'U5'}); 'U5' alone -> ('A0', {'U5'})."""
    parts = name.split('+')
    flags = {p for p in parts if p in FLAGS}
    rest = '+'.join(p for p in parts if p not in FLAGS) or 'A0'
    return rest, flags


def _filled(seq, keys):
    d = _FILLED.setdefault(id(seq), {})
    for v in keys:
        if v not in d:
            d[v] = _AG.c2_fill(np.asarray(seq[v]))
    return d


class SkelDS:
    def __init__(self, items, seq, F, train, aug='A0'):
        name, self.flags = parse_aug(aug)
        self.items, self.F, self.train, self.aug = items, F, train, AUGS[name]
        self.seq = _filled(seq, sorted({v for v, _ in items})) if 'C2' in self.flags else seq
        self.K = F.get('n_classes') or len(C.CLASSES)
        self.rate = None
        if 'ASM' in self.flags and train:            # per-joint missing rate of the training windows
            miss = np.zeros(17); n = 0
            for v, idx in items[::max(1, len(items) // 2000)]:
                w = np.asarray(self.seq[v][idx])
                miss += (w[..., 2] <= 0).sum(0); n += len(w)
            self.rate = miss / max(1, n)

    def __len__(self):
        return len(self.items)

    def _get(self, i):
        v, idx = self.items[i]
        return self.seq[v][idx].copy(), self.F['videos'][v]['cls']   # (T,17,3): x, y (box-normalised), confidence

    def _view(self, x):
        if self.train:
            x = self.aug(base_aug(x))
            if self.rate is not None:
                x = _AG.asm_mask(x, self.rate)
        if 'C1' in self.flags:
            x = _AG.c1_body_norm(x)
        elif 'U5' in self.flags:
            x = _AG.u5_upper_norm(x)
        return x

    def __getitem__(self, i):
        import torch
        x, y = self._get(i)
        if self.train and 'MIX' in self.flags:        # soft label (CrossEntropyLoss accepts class probabilities)
            t = np.zeros(self.K, np.float32); t[y] = 1
            if random.random() < 0.5:
                xb, yb = self._get(random.randrange(len(self.items)))
                x, lam = _AG.jmda_mix(self._view(x), self._view(xb))
                t = lam * t; t[yb] += 1 - lam
            else:
                x = self._view(x)
            y = torch.from_numpy(t)
        else:
            x = self._view(x)
        x = torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)).permute(2, 0, 1).contiguous()
        return x, y, i


# =============================================================================== models
def build(name):
    import torch
    import torch.nn as nn
    if name in ('ST-GCN', 'CTR-GCN'):
        return SK.build_model(name)

    A0 = torch.tensor(SK.graph_A())                   # (3,17,17)

    class DataBN(nn.Module):
        def __init__(s, C=3, V=17):
            super().__init__()
            s.bn = nn.BatchNorm1d(C * V)

        def forward(s, x):
            N, Cc, T, V = x.shape
            x = x.permute(0, 3, 1, 2).reshape(N, V * Cc, T)
            return s.bn(x).reshape(N, V, Cc, T).permute(0, 2, 3, 1).contiguous()

    # ------------------------------------------------ ST-GCN++ (PYSKL: adaptive 'init' graph + multi-scale TCN)
    class AGCN(nn.Module):
        def __init__(s, cin, cout):
            super().__init__()
            s.K = A0.shape[0]
            s.A = nn.Parameter(A0.clone())             # initialised with the skeleton graph, fully learnable
            s.conv = nn.Conv2d(cin, cout * s.K, 1)
            s.bn = nn.BatchNorm2d(cout)
            s.res = nn.Identity() if cin == cout else nn.Sequential(nn.Conv2d(cin, cout, 1), nn.BatchNorm2d(cout))
            s.relu = nn.ReLU(inplace=True)

        def forward(s, x):
            N, _, T, V = x.shape
            y = s.conv(x).view(N, s.K, -1, T, V)
            y = torch.einsum('nkctv,kwv->nctw', y, s.A)
            return s.relu(s.bn(y) + s.res(x))

    class MSTCN(nn.Module):
        def __init__(s, cin, cout, stride=1):
            super().__init__()
            nb = 6
            mids = [cout // nb] * (nb - 1) + [cout - (cout // nb) * (nb - 1)]
            br = []
            for d, m in zip((1, 2, 3, 4), mids[:4]):
                br.append(nn.Sequential(nn.Conv2d(cin, m, 1), nn.BatchNorm2d(m), nn.ReLU(inplace=True),
                                        nn.Conv2d(m, m, (3, 1), (stride, 1), (d, 0), (d, 1))))
            br.append(nn.Sequential(nn.Conv2d(cin, mids[4], 1), nn.BatchNorm2d(mids[4]), nn.ReLU(inplace=True),
                                    nn.MaxPool2d((3, 1), (stride, 1), (1, 0))))
            br.append(nn.Conv2d(cin, mids[5], 1, (stride, 1)))
            s.br = nn.ModuleList(br)
            s.out = nn.Sequential(nn.BatchNorm2d(cout), nn.ReLU(inplace=True), nn.Conv2d(cout, cout, 1),
                                  nn.BatchNorm2d(cout))

        def forward(s, x):
            return s.out(torch.cat([b(x) for b in s.br], 1))

    class PPBlock(nn.Module):
        def __init__(s, cin, cout, stride=1, residual=True):
            super().__init__()
            s.gcn, s.tcn = AGCN(cin, cout), MSTCN(cout, cout, stride)
            if not residual:
                s.res = lambda x: 0
            elif cin == cout and stride == 1:
                s.res = nn.Identity()
            else:
                s.res = nn.Sequential(nn.Conv2d(cin, cout, 1, (stride, 1)), nn.BatchNorm2d(cout))
            s.relu = nn.ReLU(inplace=True)

        def forward(s, x):
            return s.relu(s.tcn(s.gcn(x)) + s.res(x))

    class STGCNpp(nn.Module):
        def __init__(s):
            super().__init__()
            s.bn = DataBN()
            cfg = [(3, 64, 1, False), (64, 64, 1, True), (64, 64, 1, True), (64, 64, 1, True),
                   (64, 128, 2, True), (128, 128, 1, True), (128, 128, 1, True),
                   (128, 256, 2, True), (256, 256, 1, True), (256, 256, 1, True)]
            s.blocks = nn.ModuleList(PPBlock(*c) for c in cfg)
            s.drop = nn.Dropout(0.1)
            s.fc = nn.Linear(256, 6)

        def forward(s, x):
            x = s.bn(x)
            for b in s.blocks:
                x = b(x)
            return s.fc(s.drop(x.mean((2, 3))))

    # ------------------------------------------------ PoseC3D (PYSKL: SlowOnly-R50 pose variant on joint heatmaps)
    class Bottleneck3d(nn.Module):
        def __init__(s, cin, planes, stride_s=1, inflate=False):
            super().__init__()
            kt = 3 if inflate else 1
            s.c1 = nn.Sequential(nn.Conv3d(cin, planes, (kt, 1, 1), padding=(kt // 2, 0, 0), bias=False),
                                 nn.BatchNorm3d(planes), nn.ReLU(inplace=True))
            s.c2 = nn.Sequential(nn.Conv3d(planes, planes, (1, 3, 3), (1, stride_s, stride_s), (0, 1, 1), bias=False),
                                 nn.BatchNorm3d(planes), nn.ReLU(inplace=True))
            s.c3 = nn.Sequential(nn.Conv3d(planes, planes * 4, 1, bias=False), nn.BatchNorm3d(planes * 4))
            s.down = None
            if stride_s != 1 or cin != planes * 4:
                s.down = nn.Sequential(nn.Conv3d(cin, planes * 4, 1, (1, stride_s, stride_s), bias=False),
                                       nn.BatchNorm3d(planes * 4))
            s.relu = nn.ReLU(inplace=True)

        def forward(s, x):
            r = x if s.down is None else s.down(x)
            return s.relu(s.c3(s.c2(s.c1(x))) + r)

    class PoseC3D(nn.Module):
        """Input (N,3,T,17) box-normalised keypoints -> joint heatmap volume (N,17,T/2,56,56) built on the GPU."""

        def __init__(s):
            super().__init__()
            S = CFG['HM_SIZE']
            g = torch.linspace(-CFG['HM_RANGE'], CFG['HM_RANGE'], S)
            s.register_buffer('gx', g.view(1, 1, 1, 1, S))
            s.register_buffer('gy', g.view(1, 1, 1, S, 1))
            s.sig = CFG['HM_SIGMA'] * (2 * CFG['HM_RANGE'] / (S - 1))     # sigma in grid cells -> coordinates
            s.stem = nn.Sequential(nn.Conv3d(17, 32, (1, 7, 7), padding=(0, 3, 3), bias=False),
                                   nn.BatchNorm3d(32), nn.ReLU(inplace=True))
            layers, cin = [], 32
            for planes, n, inflate in ((32, 4, False), (64, 6, True), (128, 3, True)):
                for i in range(n):
                    layers.append(Bottleneck3d(cin, planes, 2 if i == 0 else 1, inflate))
                    cin = planes * 4
            s.body = nn.Sequential(*layers)
            s.drop = nn.Dropout(0.5)
            s.fc = nn.Linear(cin, 6)

        def heatmaps(s, x):
            T = x.shape[2]
            step = max(1, T // CFG['HM_FRAMES'])
            x = x[:, :, ::step][:, :, :CFG['HM_FRAMES']].float()       # (N,3,T',17)
            u = x[:, 0].permute(0, 2, 1)[..., None, None]               # (N,17,T',1,1)
            v = x[:, 1].permute(0, 2, 1)[..., None, None]
            c = x[:, 2].permute(0, 2, 1)[..., None, None]
            return c * torch.exp(-((s.gx - u) ** 2 + (s.gy - v) ** 2) / (2 * s.sig ** 2))

        def forward(s, x):
            h = s.heatmaps(x)
            return s.fc(s.drop(s.body(s.stem(h)).mean((2, 3, 4))))

    return {'ST-GCN++': STGCNpp, 'PoseC3D': PoseC3D}[name]()


def n_params(m):
    return sum(p.numel() for p in m.parameters())


# =============================================================================== units
def units(exp, augs, models, splits, seeds):
    return [dict(exp=exp, aug=a, model=m, split=s, seed=int(sd))
            for a in augs for m in models for s in splits for sd in seeds]


def unit_dir(P, u):
    return f"{P.work}/maa/runs/{u['exp']}/{u['aug']}/{u['model']}/{u['split']}/s{u['seed']}"


def unit_tag(u):
    return f"{u['exp']} {u['aug']} {u['model']} {u['split']} s{u['seed']}"


def evaluate(rdir, model, test_ds, F, cfg):
    """Video-level macro-F1 (mean of window probabilities per video), overall and per group; saves the
    per-video probabilities for the paired bootstrap."""
    import torch
    from torch.utils.data import DataLoader
    model.load_state_dict(torch.load(f'{rdir}/best.pt', map_location='cpu', weights_only=False)['model'])
    model.to('cuda')
    dl = DataLoader(test_ds, batch_size=cfg['batch'], shuffle=False, num_workers=cfg['workers'], pin_memory=True)
    yt, yp, probs, idx = C.predict(model, dl, 'cuda', True)
    wv = np.array([test_ds.items[i][0] for i in idx])
    vids = sorted(set(wv))
    vp = np.stack([probs[wv == v].mean(0) for v in vids])
    vy = np.array([F['videos'][v]['cls'] for v in vids])
    grp = np.array([F['videos'][v]['group'] for v in vids])
    own = np.array([F['videos'][v]['owner'] for v in vids])
    np.savez_compressed(f'{rdir}/video_probs.npz', videos=np.array(vids), y=vy, probs=vp, group=grp, owner=own)
    res = {}
    for tag, m in (('all', np.ones(len(vids), bool)), ('dis', grp == 'dis'), ('nondis', grp == 'nondis')):
        if m.sum():
            r = C.classification_metrics(vy[m], vp[m].argmax(1))
            res[f'vid_f1_{tag}'], res[f'vid_acc_{tag}'], res[f'n_{tag}'] = r['macro_f1'], r['acc'], int(m.sum())
            if tag == 'all':
                C.save_json(r['cm'], f'{rdir}/confusion_video.json')
    r = C.classification_metrics(yt, yp)
    res['win_f1_all'], res['win_acc_all'] = r['macro_f1'], r['acc']
    return res


def run_unit(P, F, seq, u):
    import torch
    rdir = unit_dir(P, u)
    C.seed_everything(u['seed'])
    torch.backends.cudnn.benchmark = True
    parts = C.load_split(P, u['split'])
    tr, va, te = (make_windows(parts[k], seq) for k in ('train', 'val', 'test'))
    m = u['model']
    cfg = dict(epochs=CFG['EPOCHS'], lr=CFG['LR'][m], wd=CFG['WD'], batch=CFG['BATCH'][m], accum=CFG['ACCUM'][m],
               patience=CFG['PATIENCE'], workers=CFG['WORKERS'], warmup=CFG['WARMUP'], amp=True)
    C.log(f'windows train {len(tr)} / val {len(va)} / test {len(te)}')
    t0 = time.time()
    C.train_classifier(rdir, build(m), SkelDS(tr, seq, F, True, u['aug']), SkelDS(va, seq, F, False, u['aug']), cfg)
    res = evaluate(rdir, build(m), SkelDS(te, seq, F, False, u['aug']), F, cfg)
    hist = json.load(open(f'{rdir}/history.json')) if os.path.exists(f'{rdir}/history.json') else []
    res.update(u, params=n_params(build(m)), epochs=len(hist),
               best_val_f1=max([h['val_f1'] for h in hist], default=None),
               gpu=torch.cuda.get_device_name(0), minutes=round((time.time() - t0) / 60, 1))
    C.save_json(res, f'{rdir}/DONE.json')
    C.log(f"{unit_tag(u)} finished: video macro-F1 {res['vid_f1_all']:.4f}"
          + (f" (dis {res['vid_f1_dis']:.4f})" if 'vid_f1_dis' in res else ''))


def _done_elsewhere(rdir):
    """True if DONE.json exists (fresh directory listing; Google Drive caches os.path.exists across sessions)."""
    try:
        return 'DONE.json' in os.listdir(rdir) or os.path.exists(f'{rdir}/DONE.json')
    except OSError:
        return False


def execute(P, F, seq, todo, order='forward', session_seed=None):
    """todo: list of units, or list of stages (lists of units); stages are done in order, shuffled within."""
    stages = todo if todo and isinstance(todo[0], list) else [todo]
    rnd = random.Random(session_seed if session_seed is not None else C.SESSION)
    todo = []
    for st in stages:
        st = list(st)
        if order == 'reverse':
            st = st[::-1]
        elif order == 'shuffle':
            rnd.shuffle(st)
        todo += st
    busy = []
    for u in todo:
        rdir = unit_dir(P, u)
        if os.path.exists(f'{rdir}/DONE.json'):
            continue
        if not C.claim_run(rdir):
            busy.append(unit_tag(u))
            continue
        if _done_elsewhere(rdir):                     # Drive of another session may sync late: re-check after claim
            C.release_run(rdir)
            continue
        C.log(f'===== {unit_tag(u)} =====')
        try:
            run_unit(P, F, seq, u)
        finally:
            C.release_run(rdir)
    left = [u for u in todo if not os.path.exists(f'{unit_dir(P, u)}/DONE.json')]
    C.log(f'Session finished. Open units: {len(left)} (in progress elsewhere: {len(busy)}).')
    return left


def status(P, todo):
    import pandas as pd
    rows = []
    for u in todo:
        d = unit_dir(P, u)
        st = 'done' if os.path.exists(f'{d}/DONE.json') else ('running' if os.path.exists(f'{d}/LOCK') else
                                                              ('started' if os.path.exists(f'{d}/history.json') else 'open'))
        rows.append(dict(u, status=st))
    df = pd.DataFrame(rows)
    return df.pivot_table(index=['aug', 'model'], columns='status', values='seed', aggfunc='count', fill_value=0)


# =============================================================================== results
def collect(P, exp):
    """exp: name or list of names (e.g. ['S1', 'S3'] so that the A0 baselines of S1 are included)."""
    import pandas as pd
    exps = [exp] if isinstance(exp, str) else list(exp)
    rows = []
    for e in exps:
        for f in glob.glob(f'{P.work}/maa/runs/{e}/*/*/*/*/DONE.json'):
            with open(f) as fh:
                rows.append(json.load(fh))
    if not rows:
        C.log('No finished units yet.')
        return None
    df = pd.DataFrame(rows)
    os.makedirs(f'{P.work}/maa/results', exist_ok=True)
    df.to_csv(f'{P.work}/maa/results/{"_".join(exps)}_runs.csv', index=False)
    return df


def summary(P, exp):
    """Video macro-F1 (%) per model and protocol: CS (mean of 4 folds), CS-dis, CL, CG; mean +- sd over seeds."""
    import pandas as pd
    df = collect(P, exp)
    if df is None:
        return None
    df['protocol'] = df['split'].str.replace(r'\d+$', '', regex=True)
    out = []
    for (aug, model), g in df.groupby(['aug', 'model']):
        row = dict(aug=aug, model=model)
        for prot, col, name in (('CS', 'vid_f1_all', 'CS'), ('CS', 'vid_f1_dis', 'CS-dis'),
                                ('CL', 'vid_f1_all', 'CL'), ('CG', 'vid_f1_all', 'CG')):
            h = g[g.protocol == prot]
            if h.empty or col not in h:
                continue
            per_seed = h.groupby('seed')[col].mean() * 100          # fold mean per seed
            nf = h.groupby('seed')['split'].nunique()
            full = nf.max() == (1 if prot == 'CG' else 4)
            row[name] = f"{per_seed.mean():.1f} ± {per_seed.std(ddof=1) if len(per_seed) > 1 else 0:.1f}" \
                        f" (n={len(per_seed)}{'' if full else ', incomplete'})"
        row['params_M'] = round(g['params'].iloc[0] / 1e6, 2)
        row['min_per_run'] = round(g['minutes'].median(), 1)
        out.append(row)
    s = pd.DataFrame(out)
    tag = exp if isinstance(exp, str) else '_'.join(exp)
    s.to_csv(f'{P.work}/maa/results/{tag}_summary.csv', index=False)
    return s


# ------------------------------------------------------------------ paired comparison with bootstrap CI
def _f1(y, p, k=6):
    f = []
    for c in range(k):
        if (y == c).any():
            tp = np.sum((y == c) & (p == c)); fp = np.sum((y != c) & (p == c)); fn = np.sum((y == c) & (p != c))
            f.append(2 * tp / max(1, 2 * tp + fp + fn))
    return float(np.mean(f)) if f else 0.0


def _pooled(P, exp, aug, model, splits, seed):
    """Video predictions of one seed pooled over the folds of a protocol (each video is tested once).
    exp 'FUSE': aug = 'expA@augA|expB@augB|...' -> late fusion (mean of the video probabilities) of those runs (C3)."""
    if exp == 'FUSE':
        parts = [p.split('@') for p in aug.split('|')]
        probs = [_pooled_probs(P, e, a, model, splits, seed) for e, a in parts]
        if any(p is None for p in probs):
            return None
        return {v: (y, int(np.argmax(np.mean([p[v][1] for p in probs], 0))), g) for v, (y, _, g) in probs[0].items()}
    pr = _pooled_probs(P, exp, aug, model, splits, seed)
    return None if pr is None else {v: (y, int(np.argmax(p)), g) for v, (y, p, g) in pr.items()}


def _pooled_probs(P, exp, aug, model, splits, seed):
    out = {}
    for sp in splits:
        f = f"{P.work}/maa/runs/{exp}/{aug}/{model}/{sp}/s{seed}/video_probs.npz"
        if not os.path.exists(f):
            return None
        z = np.load(f, allow_pickle=True)
        for v, y, p, g in zip(z['videos'], z['y'], z['probs'], z['group']):
            out[str(v)] = (int(y), np.asarray(p, np.float64), str(g))
    return out


def compare(P, variants, models, seeds=(0, 1, 2, 3, 4), base=('S1', 'A0'), n_boot=2000, name=None):
    """Macro-F1 difference variant - base (points), mean over the seeds both have, with a paired bootstrap over
    videos (95 % CI). Rows: CG, CS (pooled over 4 folds) all / dis / nondis, CL dis (if available)."""
    import pandas as pd
    rng = np.random.default_rng(0)
    prots = {'CG': ['CG'], 'CS': ['CS1', 'CS2', 'CS3', 'CS4'], 'CL': ['CL1', 'CL2', 'CL3', 'CL4']}
    rows = []
    for (vexp, vaug) in variants:
        for m in models:
            for prot, sps in prots.items():
                pairs = []
                for sd in seeds:
                    b, v = _pooled(P, base[0], base[1], m, sps, sd), _pooled(P, vexp, vaug, m, sps, sd)
                    if b and v:
                        pairs.append((b, v))
                if not pairs:
                    continue
                vids = sorted(pairs[0][0])
                grp = np.array([pairs[0][0][k][2] for k in vids])
                for sub in (['all', 'dis', 'nondis'] if prot != 'CG' else ['all']):
                    if prot == 'CL' and sub != 'dis':
                        continue
                    idx = np.arange(len(vids)) if sub == 'all' else np.where(grp == sub)[0]
                    if len(idx) == 0:
                        continue
                    Y = np.array([[pr[0][vids[i]][0] for i in idx] for pr in pairs])
                    PB = np.array([[pr[0][vids[i]][1] for i in idx] for pr in pairs])
                    PV = np.array([[pr[1][vids[i]][1] for i in idx] for pr in pairs])
                    fb = np.mean([_f1(Y[k], PB[k]) for k in range(len(pairs))])
                    fv = np.mean([_f1(Y[k], PV[k]) for k in range(len(pairs))])
                    d = []
                    for _ in range(n_boot):
                        r = rng.integers(0, len(idx), len(idx))
                        d.append(np.mean([_f1(Y[k, r], PV[k, r]) - _f1(Y[k, r], PB[k, r]) for k in range(len(pairs))]))
                    lo, hi = np.percentile(d, [2.5, 97.5]) * 100
                    rows.append(dict(variant=f'{vexp}:{vaug}', model=m, protocol=prot if sub == 'all' else f'{prot}-{sub}',
                                     seeds=len(pairs), videos=len(idx), base=round(fb * 100, 1), variant_f1=round(fv * 100, 1),
                                     diff=round((fv - fb) * 100, 1), ci_lo=round(lo, 1), ci_hi=round(hi, 1),
                                     significant=bool(lo > 0 or hi < 0)))
    df = pd.DataFrame(rows)
    if len(df):
        os.makedirs(f'{P.work}/maa/results', exist_ok=True)
        df.to_csv(f"{P.work}/maa/results/{name or f'compare_vs_{base[0]}_{base[1]}'}.csv", index=False)
    return df


In [ ]:
%%writefile /content/augs.py
"""
Skeleton augmentations for MAA (COCO-17, box-normalised: x right, y down, box centre = 0, box height = 1).
Input/output of every transform: float32 array (T, 17, 3) = (x, y, confidence); confidence 0 = missing joint.

Generic (comparison):
  A1  strong affine: rotation +-20 deg, scale 0.8-1.2, shear +-0.2, translation +-0.1
  A2  joint dropout (10 % of joints) and body-part dropout (one of arms / legs / head, p = 0.3)
  A3  temporal: speed perturbation 0.7-1.3 with resampling to the same length
  A4  A1 + A2 + A3

Proposed mobility-aid kinematic augmentations (applied to one sample with probability P_AID):
  U1  seated (wheelchair): static seated lower body, upper body lowered by the thigh length
  U2  occupied arm (single crutch / cane, as in CARR): one arm fixed beside the body
  U3  asymmetric gait: reduced motion of one leg, slower tempo, lateral trunk sway
  U4  aid occlusion: lower legs (footrest) or aid-side forearm missing, or noisy / swapped legs, over a time segment
  U   one of U1-U4 drawn at random (U1 may be combined with U4-legs)
  U5  upper-body-centred normalisation (a representation, used at training and test time)

Proposed mobility-aid observation (sensing) augmentations, S3c. An aid changes not only how the body moves but also
what the detection and pose pipeline delivers (S2): the annotated box contains the aid, whole frames lose the pose,
and the legs are estimated with low confidence or on the aid. Ranges are physical priors, not fitted to test users.
  O1  box inflation: the box also covers the aid -> skeleton smaller and off-centre in box coordinates, box jitter
  O2  burst frame loss: 1-3 bursts of frames without any pose, 10-60 % of the window
  O3  leg corruption: leg confidence collapses, leg joints noisy or drawn towards the aid beside / below the body
  O   O1-O3 (each with its own probability, at least one), applied to a sample with probability P_AID
  K   kinematic family U1-U3 (one of them, probability P_AID)

S4, proposed ARC (Aid-Robust Canonicalisation), flags in the variant name (train and test):
  C1  confidence-aware body anchor + view-robust body scale (replaces the box frame; successor of U5)
  C2  temporal gap filling of missing joints / frames, on the whole sequence before windowing
  C3  late fusion of a box-frame stream and a body-frame stream (post hoc, see maa.compare with exp 'FUSE')
Comparators (re-implemented, same backbones and protocol): OCC body-part occlusion, ASM adaptive spatial masking
(needs the training-set missing rate, set in maa.SkelDS), MIX JMDA-style temporal / spatial mixing with soft labels.
After U1-U3 the skeleton is re-fitted to the original keypoint extent, as the annotated box would be.
"""
import math, random
import numpy as np

NOSE, LSHO, RSHO, LELB, RELB, LWRI, RWRI, LHIP, RHIP, LKNE, RKNE, LANK, RANK = 0, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16
UPPER = list(range(0, 13))              # head, arms, hips
LEGS = [LKNE, RKNE, LANK, RANK]
FLIP = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]
P_AID = 0.5


# ----------------------------------------------------------------------------- helpers
def _valid(x):
    return x[..., 2] > 0


def _mean_pt(x, j):
    """Mean position of joint j over the valid frames (or None)."""
    v = _valid(x[:, j])
    return x[v, j, :2].mean(0) if v.any() else None


def _seg_len(x, a, b):
    v = _valid(x[:, a]) & _valid(x[:, b])
    return float(np.linalg.norm(x[v, a, :2] - x[v, b, :2], axis=1).mean()) if v.any() else None


def _extent(x):
    v = _valid(x)
    if not v.any():
        return None
    p = x[v][:, :2]
    lo, hi = p.min(0), p.max(0)
    return (lo + hi) / 2, max(hi[1] - lo[1], 1e-3)


def refit(x_new, x_old, full=None):
    """Re-centre on the original keypoint extent and re-scale towards it. The box of a seated person or of a
    person with an aid is drawn around the person *and* the aid, so the skeleton fills the box only partly:
    the scale is drawn between 'unchanged' (u = 0) and 'fills the original extent' (u = 1)."""
    e0, e1 = _extent(x_old), _extent(x_new)
    if e0 is None or e1 is None:
        return x_new
    (c0, h0), (c1, h1) = e0, e1
    u = random.uniform(0, 1) if full is None else float(full)
    k = (h0 / h1) ** u
    v = _valid(x_new)
    x_new[..., :2] = np.where(v[..., None], (x_new[..., :2] - c1) * k + c0, 0)
    return x_new


def _resample(x, factor):
    """Play the sequence `factor` times faster (factor < 1: slower) and bring it back to length T."""
    T = len(x)
    src = np.clip(np.arange(T) * factor, 0, T - 1)
    if factor < 1:                                  # slower: random start so that all phases occur
        src = src + random.uniform(0, max(0.0, (T - 1) - src[-1]))
    i0 = np.floor(src).astype(int)
    i1 = np.minimum(i0 + 1, T - 1)
    w = (src - i0)[:, None, None]
    out = x[i0] * (1 - w) + x[i1] * w
    both = _valid(x[i0]) & _valid(x[i1])
    out[..., 2] = np.where(both, out[..., 2], 0)
    out[..., :2] = np.where(both[..., None], out[..., :2], 0)
    return out.astype(np.float32)


def _drop(x, joints, t0=0, t1=None):
    x[t0:t1, joints] = 0
    return x


# ----------------------------------------------------------------------------- generic
def a1_affine(x):
    a = math.radians(random.uniform(-20, 20))
    s = random.uniform(0.8, 1.2)
    sh = random.uniform(-0.2, 0.2)
    M = np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]], np.float32) @ \
        np.array([[1, sh], [0, 1]], np.float32) * s
    t = np.random.uniform(-0.1, 0.1, 2).astype(np.float32)
    v = _valid(x)
    x[..., :2] = np.where(v[..., None], x[..., :2] @ M.T + t, 0)
    return x


def a2_dropout(x):
    m = np.random.rand(17) < 0.1
    x[:, m] = 0
    if random.random() < 0.3:
        part = random.choice([[LELB, LWRI, RELB, RWRI], LEGS, [0, 1, 2, 3, 4]])
        x[:, part] = 0
    return x


def a3_speed(x):
    return _resample(x, random.uniform(0.7, 1.3))


def a4_generic(x):
    return a2_dropout(a1_affine(a3_speed(x)))


# ----------------------------------------------------------------------------- proposed
def u1_seated(x):
    """Seated in a wheelchair, seen from an elevated camera: torso lowered and slightly leaning, thighs pointing
    towards the camera (short in the image) with the knees apart, shanks foreshortened with the feet on the
    footrest, legs static. The person fills only part of the box (the box also contains the chair)."""
    old = x.copy()
    lth = _seg_len(x, LHIP, LKNE) or _seg_len(x, RHIP, RKNE) or 0.22
    lsh = _seg_len(x, LKNE, LANK) or _seg_len(x, RKNE, RANK) or 0.22
    sho_c = (x[:, LSHO, :2] + x[:, RSHO, :2]) / 2
    hip_c = (x[:, LHIP, :2] + x[:, RHIP, :2]) / 2
    # 1. lean: shoulders and head come closer to the hips
    lean = random.uniform(0.2, 0.45)                # seated torso looks shorter (lean, slumped, seen from above)
    top = [0, 1, 2, 3, 4, LSHO, RSHO, LELB, RELB, LWRI, RWRI]
    x[:, top, 1] -= np.where(_valid(x[:, top]), lean * (sho_c[:, None, 1] - hip_c[:, None, 1]), 0)
    # 2. the whole upper body comes down by about one thigh length
    drop = lth * random.uniform(0.6, 1.0)
    x[:, UPPER, 1] += np.where(_valid(x[:, UPPER]), drop, 0)
    # 3. seated legs (identical shape over time, they move with the chair)
    f = random.uniform(0.15, 0.55)                  # thigh seen almost end-on from the elevated camera
    spread = random.uniform(0.02, 0.07)             # knees apart
    shank = lsh * random.uniform(0.5, 0.8)          # feet forward on the footrest
    inward = random.uniform(-0.01, 0.03)
    hw = np.abs(x[:, LHIP, 0] - x[:, RHIP, 0]).mean() / 2 + 1e-6
    for hip, kne, ank in ((LHIP, LKNE, LANK), (RHIP, RKNE, RANK)):
        hv = _valid(x[:, hip])
        if not hv.any():
            x[:, [kne, ank]] = 0
            continue
        out = np.sign((x[hv, hip, 0] - hip_c[hv, 0]).mean()) or 1.0
        kn = x[:, hip, :2] + np.array([out * spread, lth * f], np.float32)
        an = kn + np.array([-out * inward, shank], np.float32)
        jit = lambda: np.random.normal(0, 0.003, (len(x), 2)).astype(np.float32)
        x[:, kne, :2], x[:, ank, :2] = kn + jit(), an + jit()
        x[:, kne, 2] = np.where(hv, x[:, hip, 2], 0)
        x[:, ank, 2] = np.where(hv, x[:, hip, 2] * 0.8, 0)
        x[~hv, kne, :2] = 0
        x[~hv, ank, :2] = 0
    del hw
    return refit(x, old, full=random.uniform(0.0, 0.2))


def u2_arm(x):
    old = x.copy()
    sides = [(LSHO, LELB, LWRI, LHIP), (RSHO, RELB, RWRI, RHIP)]
    random.shuffle(sides)
    sides = sides[:1]                              # CARR: crutch and cane users hold a single aid
    hip_c = (x[:, LHIP, :2] + x[:, RHIP, :2]) / 2
    out_w = random.uniform(0.05, 0.15)
    down_w = random.uniform(0.0, 0.12)
    for k, (sho, elb, wri, hip) in enumerate(sides):
        ok = _valid(x[:, sho]) & _valid(x[:, hip])
        if not ok.any():
            continue
        # outward = away from the body centre on the image (depends on whether the person faces the camera)
        outward = 1.0 if (x[ok, hip, 0] - hip_c[ok, 0]).mean() >= 0 else -1.0
        w = x[:, hip, :2] + np.array([outward * out_w, down_w], np.float32)
        e = (x[:, sho, :2] + w) / 2 + np.array([outward * 0.03, 0], np.float32)
        noise = lambda: np.random.normal(0, 0.004, (len(x), 2)).astype(np.float32)
        x[ok, wri, :2], x[ok, elb, :2] = (w + noise())[ok], (e + noise())[ok]
        x[ok, wri, 2] = x[ok, elb, 2] = np.minimum(x[ok, sho, 2], x[ok, hip, 2])
    return refit(x, old)


def u3_gait(x):
    old = x.copy()
    x = _resample(x, random.uniform(0.6, 0.85))           # slower tempo
    hip_c = (x[:, LHIP, :2] + x[:, RHIP, :2]) / 2
    side = random.choice([(LKNE, LANK, LHIP), (RKNE, RANK, RHIP)])
    a = random.uniform(0.3, 0.8)                           # remaining motion of the affected leg
    for j in side[:2]:
        v = _valid(x[:, j]) & _valid(x[:, side[2]])
        if v.any():
            rel = x[:, j, :2] - x[:, side[2], :2]
            m = rel[v].mean(0)
            x[v, j, :2] = x[v, side[2], :2] + m + (rel[v] - m) * a
    amp = random.uniform(0.015, 0.045)                     # lateral trunk sway with each step
    per = random.uniform(15, 30)
    ph = random.uniform(0, 2 * math.pi)
    sway = amp * np.sin(2 * math.pi * np.arange(len(x)) / per + ph)
    up = [0, 1, 2, 3, 4, LSHO, RSHO, LELB, RELB, LWRI, RWRI]
    x[:, up, 0] += np.where(_valid(x[:, up]), sway[:, None], 0)
    return refit(x, old)


def u4_occlusion(x, legs=None):
    T = len(x)
    if legs is None:
        legs = random.random() < 0.5
    if random.random() < 0.5:
        t0, t1 = 0, T
    else:
        L = random.randint(T // 4, T)
        t0 = random.randint(0, T - L)
        t1 = t0 + L
    if legs:
        if random.random() < 0.4:                  # aid confuses the pose estimator: noisy, crossed legs
            v = _valid(x[t0:t1, LEGS])
            x[t0:t1, LEGS, :2] += np.where(v[..., None], np.random.normal(0, 0.03, (t1 - t0, 4, 2)), 0).astype(np.float32)
            if random.random() < 0.5:
                x[t0:t1, [LKNE, RKNE, LANK, RANK]] = x[t0:t1, [RKNE, LKNE, RANK, LANK]]
            return x
        return _drop(x, random.choice([[LANK, RANK], LEGS]), t0, t1)
    return _drop(x, random.choice([[LWRI], [RWRI], [LELB, LWRI], [RELB, RWRI]]), t0, t1)


def u_any(x, which=('U1', 'U2', 'U3', 'U4')):
    if random.random() >= P_AID:
        return x
    k = random.choice(which)
    if k == 'U1':
        x = u1_seated(x)
        if random.random() < 0.3:
            x = u4_occlusion(x, legs=True)
        return x
    return {'U2': u2_arm, 'U3': u3_gait, 'U4': u4_occlusion}[k](x)


def u5_upper_norm(x):
    """Upper-body-centred normalisation of a whole window: origin = mid-shoulder of the first valid frame,
    unit = median shoulder width x 3 (about the box height of a standing adult). Falls back to the input."""
    v = _valid(x[:, LSHO]) & _valid(x[:, RSHO])
    if v.sum() < 2:
        return x
    mid = (x[v, LSHO, :2] + x[v, RSHO, :2]) / 2
    sw = np.median(np.linalg.norm(x[v, LSHO, :2] - x[v, RSHO, :2], axis=1))
    if sw < 0.03:
        return x
    o, s = mid[0], 1.0 / (3 * sw)
    vv = _valid(x)
    x[..., :2] = np.where(vv[..., None], (x[..., :2] - o) * s, 0)
    return x


# ----------------------------------------------------------------------------- proposed: observation family
def o1_box(x):
    """Box inflated by the aid: x_box' = (x_box - d) / h with h = new/old box height and d = centre shift (in old
    box heights), plus a small smooth per-frame box jitter (box-normalised coordinates, so no pixel geometry needed)."""
    T = len(x)
    h = random.uniform(1.0, 1.45)
    dx = random.choice([-1, 1]) * random.uniform(0.0, 0.25)          # aid beside the body (crutch, cane, wheel)
    dy = (h - 1) * random.uniform(-0.5, 0.5)                          # box grows up / down
    jit = np.cumsum(np.random.normal(0, 0.004, (T, 2)), 0).astype(np.float32)
    jit -= jit.mean(0)
    hj = 1 + np.cumsum(np.random.normal(0, 0.003, T)).astype(np.float32)
    d = np.array([dx, dy], np.float32) + jit
    v = _valid(x)
    x[..., :2] = np.where(v[..., None], (x[..., :2] - d[:, None]) / (h * hj[:, None, None]), 0)
    return x


def o2_frameloss(x):
    """Whole frames without a pose, in 1-3 bursts covering 10-60 % of the window (at least 25 % of frames kept)."""
    T = len(x)
    total = int(T * random.uniform(0.1, 0.6))
    n = random.randint(1, 3)
    lens = np.random.multinomial(total, [1 / n] * n)
    for L in lens:
        if L < 1:
            continue
        t0 = random.randint(0, max(0, T - L))
        x[t0:t0 + L] = 0
    return x                                                        # o_any falls back if too little is left


def o3_legs(x):
    """Legs estimated badly: confidence collapse and noise, or (30 %) leg joints pulled towards the aid."""
    v = _valid(x[:, LEGS])
    x[:, LEGS, 2] = np.where(v, x[:, LEGS, 2] * random.uniform(0.1, 0.5), 0)
    sd = random.uniform(0.01, 0.04)
    x[:, LEGS, :2] += np.where(v[..., None], np.random.normal(0, sd, (len(x), 4, 2)), 0).astype(np.float32)
    if random.random() < 0.3:
        hv = _valid(x[:, LHIP]) & _valid(x[:, RHIP])
        if hv.any():
            hip_c = (x[:, LHIP, :2] + x[:, RHIP, :2]) / 2
            tgt = hip_c + np.array([random.choice([-1, 1]) * random.uniform(0.05, 0.2), random.uniform(0.2, 0.45)],
                                   np.float32)
            w = random.uniform(0.3, 0.8)
            ok = v & hv[:, None]
            x[:, LEGS, :2] = np.where(ok[..., None], (1 - w) * x[:, LEGS, :2] + w * tgt[:, None], x[:, LEGS, :2])
    return x


def o_any(x, which=('O1', 'O2', 'O3'), p=None):
    if random.random() >= (P_AID if p is None else p):
        return x
    fn = {'O1': (o1_box, 0.7), 'O2': (o2_frameloss, 0.5), 'O3': (o3_legs, 0.5)}
    pick = [k for k in which if random.random() < fn[k][1]] or [random.choice(which)]
    x0 = x.copy()
    for k in ('O3', 'O1', 'O2'):                                     # legs, then box, then frame loss
        if k in pick:
            x = fn[k][0](x)
    return x if _valid(x).any(1).sum() >= len(x) // 4 else x0


def k_any(x):
    return u_any(x, ('U1', 'U2', 'U3'))


def single(name):
    return {'U1': u1_seated, 'U2': u2_arm, 'U3': u3_gait, 'U4': u4_occlusion}[name]


AUGS = {
    'A0': lambda x: x,
    'A1': a1_affine, 'A2': a2_dropout, 'A3': a3_speed, 'A4': a4_generic,
    'U1': lambda x: u_any(x, ('U1',)), 'U2': lambda x: u_any(x, ('U2',)),
    'U3': lambda x: u_any(x, ('U3',)), 'U4': lambda x: u_any(x, ('U4',)),
    'U': u_any,
    'A4+U': lambda x: u_any(a4_generic(x)),
    'A4+U1': lambda x: u_any(a4_generic(x), ('U1',)),
    'U1+U4': lambda x: u_any(x, ('U1', 'U4')),
    # S3c: observation (sensing) family and its combination with the kinematic family
    'O': o_any, 'K': k_any,
    'K+O': lambda x: o_any(k_any(x)),
    'A4+O': lambda x: o_any(a4_generic(x)),
    'A4+K': lambda x: k_any(a4_generic(x)),
    'A4+K+O': lambda x: o_any(k_any(a4_generic(x))),
    'A4+O1': lambda x: o_any(a4_generic(x), ('O1',)),
    'A4+O2': lambda x: o_any(a4_generic(x), ('O2',)),
    'A4+O3': lambda x: o_any(a4_generic(x), ('O3',)),
}


# =============================================================================== S4: ARC (Aid-Robust Canonicalisation)
TORSO = [LSHO, RSHO, LHIP, RHIP]


def c1_body_norm(x, k_sw=3.0, k_tl=3.3, n0=8, min_conf=0.3):
    """C1: confidence-aware body anchor that does not depend on the box.
    Origin: confidence-weighted mid-shoulder (fallback mid-hip, then nose) averaged over the first n0 frames in which
    it is reliable, so the translation of the person inside the window is kept.
    Scale: the larger of 3 x shoulder width and 3.3 x torso length (median over reliable frames) - shoulder width
    collapses in side view, torso length collapses when seated / leaning, the maximum is robust to both."""
    c = x[..., 2]
    ok = lambda *j: np.all([c[:, i] >= min_conf for i in j], axis=0)
    sho, hip = ok(LSHO, RSHO), ok(LHIP, RHIP)
    msho = (x[:, LSHO, :2] + x[:, RSHO, :2]) / 2
    mhip = (x[:, LHIP, :2] + x[:, RHIP, :2]) / 2
    if sho.any():
        idx = np.where(sho)[0][:n0]
        w = np.minimum(c[idx, LSHO], c[idx, RSHO])[:, None]
        o = (msho[idx] * w).sum(0) / w.sum()
    elif hip.any():
        idx = np.where(hip)[0][:n0]
        o = mhip[idx].mean(0)
    elif (c[:, NOSE] >= min_conf).any():
        o = x[c[:, NOSE] >= min_conf, NOSE, :2][:n0].mean(0)
    else:
        return x
    cand = []
    if sho.sum() >= 2:
        cand.append(k_sw * np.median(np.linalg.norm(x[sho, LSHO, :2] - x[sho, RSHO, :2], axis=1)))
    both = sho & hip
    if both.sum() >= 2:
        cand.append(k_tl * np.median(np.linalg.norm(msho[both] - mhip[both], axis=1)))
    s = max(cand) if cand else 0.0
    if s < 0.1:                                      # unreliable body size: keep the box scale, only re-centre
        s = 1.0
    v = c > 0
    x[..., :2] = np.where(v[..., None], (x[..., :2] - o) / s, 0)
    return x


def c2_fill(seq, max_gap=64, max_hold=8, fill_conf=0.05):
    """C2: temporal gap filling of a whole sequence (T,17,3), before windowing / augmentation (train and test).
    Interior gaps of a joint up to max_gap frames are linearly interpolated, gaps at the start / end are held for
    up to max_hold frames. Filled joints get a low confidence so that the network can tell them apart."""
    x = seq.copy()
    T = len(x)
    t = np.arange(T)
    for j in range(x.shape[1]):
        v = x[:, j, 2] > 0
        if v.all() or v.sum() < 2:
            continue
        tv = t[v]
        nxt = np.searchsorted(tv, t)                         # index of next valid frame
        prv = nxt - 1
        miss = ~v
        inner = miss & (nxt < len(tv)) & (prv >= 0)
        gap = np.where(inner, tv[np.clip(nxt, 0, len(tv) - 1)] - tv[np.clip(prv, 0, len(tv) - 1)] - 1, 10 ** 6)
        fill_in = inner & (gap <= max_gap)
        head = miss & (prv < 0) & (tv[0] - t <= max_hold)
        tail = miss & (nxt >= len(tv)) & (t - tv[-1] <= max_hold)
        for d in (0, 1):
            interp = np.interp(t, tv, x[v, j, d])
            x[fill_in | head | tail, j, d] = interp[fill_in | head | tail]
        x[fill_in | head | tail, j, 2] = fill_conf
    return x


# ----------------------------------------------------------------------------- comparators (re-implemented)
PARTS = {'head': [0, 1, 2, 3, 4], 'larm': [LSHO, LELB, LWRI], 'rarm': [RSHO, RELB, RWRI],
         'lleg': [LHIP, LKNE, LANK], 'rleg': [RHIP, RKNE, RANK], 'torso': [LSHO, RSHO, LHIP, RHIP]}


def occ_parts(x, p=0.5):
    """OCC: body-part occlusion augmentation (Mathe et al., Sensors 2025): one or two body parts removed from the
    whole window."""
    if random.random() < p:
        for part in random.sample(list(PARTS), random.choice([1, 2])):
            x[:, PARTS[part]] = 0
    return x


def asm_mask(x, rate, p=0.5, ratio=0.15):
    """ASM: adaptive spatial masking (Wang et al., 2023): joints are masked with a probability proportional to how
    often they are missing in the training data (rate: per-joint missing rate of the training set)."""
    if random.random() < p:
        q = np.clip(ratio * rate / max(rate.mean(), 1e-6), 0, 0.6)
        x[:, np.random.rand(17) < q] = 0
    return x


def jmda_mix(xa, xb):
    """MIX: JMDA-style mixing (Xiang & Wang, ACM TOMM 2024), adapted to 2D poses. TemporalMix: both sequences are
    resized in time and concatenated; SpatialMix: upper body of A with lower body of B (B aligned at the hip centre).
    Returns the mixed sequence and the weight lam of sample A for the label."""
    T = len(xa)
    if random.random() < 0.5:
        lam = random.uniform(0.3, 0.7)
        ta = max(1, int(round(lam * T)))
        ia = np.linspace(0, T - 1, ta).round().astype(int)
        ib = np.linspace(0, T - 1, T - ta).round().astype(int)
        return np.concatenate([xa[ia], xb[ib]], 0).astype(np.float32), ta / T
    lower = [LHIP, RHIP, LKNE, RKNE, LANK, RANK]
    out = xa.copy()
    va = _valid(xa[:, LHIP]) & _valid(xa[:, RHIP])
    vb = _valid(xb[:, LHIP]) & _valid(xb[:, RHIP])
    if va.any() and vb.any():
        ha = ((xa[va, LHIP, :2] + xa[va, RHIP, :2]) / 2).mean(0)
        hb = ((xb[vb, LHIP, :2] + xb[vb, RHIP, :2]) / 2).mean(0)
        sh = xb[:, lower].copy()
        sh[..., :2] = np.where(_valid(sh)[..., None], sh[..., :2] - hb + ha, 0)
        out[:, lower] = sh
    return out, 11 / 17


AUGS.update({
    'OCC': occ_parts, 'A4+OCC': lambda x: occ_parts(a4_generic(x)),
})


In [ ]:
%%writefile /content/maa_safer.py
"""
S9: external validation on SAFER-Activities (CC BY-NC-SA 4.0, Lamsal et al., 2026).

Conversion (once, by the first session; the others wait):
  * 2D poses (YOLOv8x + ViTPose-H, COCO-17) and boxes from pose_bboxes/3d_keypoints_pickle_ntu_format.zip
  * action segments from the CSVs (Action, Start, End) at 25 fps; every (camera view, segment) becomes one sample
  * classes = actions present in both the non-wheelchair and the wheelchair subset with at least MIN_TEST_SEG
    segments in the wheelchair subject-test split
  * keypoints normalised exactly as in CARR: box centre = 0, box height = 1, confidence as third channel
Protocols:
  SZ  zero-shot to wheelchair: train on non-wheelchair subjects that are NOT in the wheelchair test split;
      validation = half of the non-wheelchair test subjects; test = wheelchair subject-test split ('dis')
      + the other half of the non-wheelchair test subjects ('nondis', check that nothing is lost for them)
  SO  oracle: train and validate on the wheelchair subject-train split, test on the wheelchair subject-test split
"""
import os, re, csv, glob, json, time, gc, random, collections, zipfile
import numpy as np

import maa
from carr import common as C

FPS = 25.0
MIN_TEST_SEG = 10            # minimum segments (all camera views) of a class in the wheelchair test split
MAX_SEG_SEC = 20.0           # longer segments: the middle 20 s are kept
MIN_SEG_FRAMES = 8
TRAIN_WIN_PER_SEG = 2        # windows per segment for training (evenly spaced), evaluation: up to 8
EVAL_WIN_PER_SEG = 8


def _secs(t):
    h, m, s = t.strip().split(':')
    return int(h) * 3600 + int(m) * 60 + float(s)


def _labels(safer):
    lab = {}
    for f in glob.glob(f'{safer}/raw/Safer-Activities-Full-Dataset/*/CSVs/*.csv'):
        rows = list(csv.reader(open(f, encoding='utf-8-sig')))[1:]
        segs = []
        for r in rows:
            if len(r) >= 4 and r[2].strip() and r[3].strip():
                try:
                    segs.append((r[1].strip().lower(), _secs(r[2]), _secs(r[3])))
                except ValueError:
                    pass
        lab[os.path.basename(f)[:-4]] = segs
    return lab


def _recording(frame_dir):
    return re.sub(r'_d\d+$', '', frame_dir.replace('.mp4', ''))


def _subject(frame_dir):
    m = re.search(r'_(p\d+)_d\d+', frame_dir)
    return m.group(1) if m else frame_dir


def _normalise(kp, sc, bb, fmt):
    """kp (T,17,2) pixels, sc (T,17), bb (T,4) -> (T,17,3) box-normalised like CARR."""
    bb = bb.astype(np.float32)
    if fmt == 'xywh':
        x1, y1, w, h = bb.T
    else:
        x1, y1, x2, y2 = bb.T
        w, h = x2 - x1, y2 - y1
    cx, cy = x1 + w / 2, y1 + h / 2
    ok = h > 2
    hh = np.where(ok, h, 1.0)
    valid = (sc > 0.05) & ok[:, None]
    xs = (kp[..., 0] - cx[:, None]) / hh[:, None]
    ys = (kp[..., 1] - cy[:, None]) / hh[:, None]
    c = np.clip(sc, 0, 1)
    return np.stack([np.where(valid, xs, 0), np.where(valid, ys, 0), np.where(valid, c, 0)], -1).astype(np.float16)


def convert(safer, out_file, work='/content/safer_pkl'):
    import pickle
    lab = _labels(safer)
    os.makedirs(work, exist_ok=True)
    zf = f'{safer}/pose_bboxes/3d_keypoints_pickle_ntu_format.zip'
    with zipfile.ZipFile(zf) as z:
        for n in z.namelist():
            if n.endswith(('aic_normal_dataset_with_3d.pkl', 'aic_wheelchair_dataset_with_3d.pkl')) and \
                    not os.path.exists(f'{work}/{os.path.basename(n)}'):
                C.log(f'extract {n}')
                with z.open(n) as src, open(f'{work}/{os.path.basename(n)}', 'wb') as dst:
                    while True:
                        b = src.read(1 << 24)
                        if not b:
                            break
                        dst.write(b)
    # pass 1: class counts in the wheelchair test split, to choose the classes
    seg_meta = []
    for subset, fn in (('normal', 'aic_normal_dataset_with_3d.pkl'), ('wheelchair', 'aic_wheelchair_dataset_with_3d.pkl')):
        C.log(f'read {fn}')
        d = pickle.load(open(f'{work}/{fn}', 'rb'))
        split_of = {}
        for k in ('sub_train', 'sub_test'):
            for v in d['split'][k]:
                split_of[v] = k
        for a in d['annotations']:
            fd = a['frame_dir']
            if fd not in split_of:
                continue
            rec = _recording(fd)
            if rec not in lab:
                continue
            T = int(a['total_frames'])
            kp = np.asarray(a['keypoint'])[0]
            sc = np.asarray(a['keypoint_score'])[0]
            bb = np.asarray(a['bboxes'])
            for i, (act, t0, t1) in enumerate(lab[rec]):
                f0, f1 = int(round(t0 * FPS)), min(T, int(round(t1 * FPS)))
                if f1 - f0 < MIN_SEG_FRAMES:
                    continue
                if (f1 - f0) > MAX_SEG_SEC * FPS:
                    mid = (f0 + f1) // 2
                    f0, f1 = int(mid - MAX_SEG_SEC * FPS / 2), int(mid + MAX_SEG_SEC * FPS / 2)
                x = _normalise(kp[f0:f1], sc[f0:f1], bb[f0:f1], a.get('bbox_format', 'xyxy'))
                if (x[..., 2] > 0).mean() < 0.2:       # person mostly not detected (blacked-out gap)
                    continue
                seg_meta.append(dict(key=f'{fd}#{i:04d}', subset=subset, split=split_of[fd], video=fd,
                                     subject=_subject(fd), action=act, x=x))
        del d
        gc.collect()
    cnt_wc_test = collections.Counter(s['action'] for s in seg_meta if s['subset'] == 'wheelchair' and s['split'] == 'sub_test')
    acts_n = {s['action'] for s in seg_meta if s['subset'] == 'normal'}
    acts_w = {s['action'] for s in seg_meta if s['subset'] == 'wheelchair'}
    shared = sorted(acts_n & acts_w)
    classes = [a for a in shared if cnt_wc_test[a] >= MIN_TEST_SEG]
    C.log(f'shared classes {shared}; kept (>= {MIN_TEST_SEG} test segments): {classes}')
    keep = [s for s in seg_meta if s['action'] in classes]
    obj = dict(classes=classes, shared=shared, wc_test_counts=dict(cnt_wc_test), fps=FPS,
               seq={s['key']: s['x'] for s in keep},
               meta={s['key']: {k: s[k] for k in ('subset', 'split', 'video', 'subject', 'action')} for s in keep})
    C.save_pickle(obj, out_file)
    C.log(f'{len(keep)} segments saved to {out_file}')
    return obj


def load_or_convert(safer, out_file):
    """The first session converts (lock), the others wait until the file exists."""
    if not os.path.exists(out_file):
        ldir = os.path.dirname(out_file) + '/convert_lock'
        if C.claim_run(ldir):
            try:
                if not os.path.exists(out_file):
                    convert(safer, out_file)
            finally:
                C.release_run(ldir)
        while not os.path.exists(out_file):
            C.log('another session is converting SAFER, waiting 60 s...')
            time.sleep(60)
    return C.load_pickle(out_file)


# ----------------------------------------------------------------------------- data for the trainer
def prepare(obj):
    classes = obj['classes']
    K = len(classes)
    seq = {k: v.astype(np.float32) for k, v in obj['seq'].items()}
    F = {'videos': {k: dict(cls=classes.index(m['action']), group='dis' if m['subset'] == 'wheelchair' else 'nondis',
                            owner=m['subject'], split=m['split'], subset=m['subset'])
                    for k, m in obj['meta'].items()}}
    meta = obj['meta']
    wc_test_subj = {m['subject'] for m in meta.values() if m['subset'] == 'wheelchair' and m['split'] == 'sub_test'}
    n_test_subj = sorted({m['subject'] for m in meta.values() if m['subset'] == 'normal' and m['split'] == 'sub_test'}
                         - wc_test_subj)
    val_subj = set(n_test_subj[::2])
    keys = list(meta)
    sz = dict(train=[k for k in keys if meta[k]['subset'] == 'normal' and meta[k]['split'] == 'sub_train'
                     and meta[k]['subject'] not in wc_test_subj],
              val=[k for k in keys if meta[k]['subset'] == 'normal' and meta[k]['split'] == 'sub_test'
                   and meta[k]['subject'] in val_subj],
              test=[k for k in keys if (meta[k]['subset'] == 'wheelchair' and meta[k]['split'] == 'sub_test') or
                    (meta[k]['subset'] == 'normal' and meta[k]['split'] == 'sub_test'
                     and meta[k]['subject'] not in val_subj and meta[k]['subject'] not in wc_test_subj)])
    wtr = sorted({meta[k]['subject'] for k in keys if meta[k]['subset'] == 'wheelchair' and meta[k]['split'] == 'sub_train'})
    wval = {wtr[0]} if wtr else set()
    so = dict(train=[k for k in keys if meta[k]['subset'] == 'wheelchair' and meta[k]['split'] == 'sub_train'
                     and meta[k]['subject'] not in wval],
              val=[k for k in keys if meta[k]['subset'] == 'wheelchair' and meta[k]['split'] == 'sub_train'
                   and meta[k]['subject'] in wval],
              test=[k for k in keys if meta[k]['subset'] == 'wheelchair' and meta[k]['split'] == 'sub_test'])
    info = dict(classes=classes, wc_test_subjects=sorted(wc_test_subj), normal_val_subjects=sorted(val_subj),
                wheelchair_val_subject=sorted(wval))
    F['n_classes'] = K
    return seq, F, {'SZ': sz, 'SO': so}, K, info


def windows(keys, seq, per_seg):
    W, S = maa.SK.CFG['WIN'], maa.SK.CFG['STRIDE']
    items = []
    for k in keys:
        L = len(seq[k])
        if L >= W:
            starts = list(range(0, L - W + 1, S))
            if len(starts) > per_seg:
                starts = [starts[int(round(i))] for i in np.linspace(0, len(starts) - 1, per_seg)]
            for s in starts:
                items.append((k, np.arange(s, s + W)))
        else:
            items.append((k, np.arange(W) % L))
    return items


def _patch_metrics(K):
    from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

    def cm(y_true, y_pred):
        return dict(acc=float(accuracy_score(y_true, y_pred)),
                    macro_f1=float(f1_score(y_true, y_pred, average='macro', labels=list(range(K)), zero_division=0)),
                    cm=confusion_matrix(y_true, y_pred, labels=list(range(K))).tolist())
    C.classification_metrics = cm


def build_k(name, K):
    import torch.nn as nn
    net = maa.build(name)
    last = None
    for mod_name, mod in net.named_modules():
        if isinstance(mod, nn.Linear):
            last = mod_name
    parent = net
    parts = last.split('.')
    for p in parts[:-1]:
        parent = getattr(parent, p)
    old = getattr(parent, parts[-1])
    setattr(parent, parts[-1], nn.Linear(old.in_features, K))
    return net


def unit_dir(P, u):
    return f"{P.work}/maa/runs/{u['exp']}/{u['aug']}/{u['model']}/{u['split']}/s{u['seed']}"


def run_unit(P, seq, F, prots, K, u):
    import torch
    rdir = unit_dir(P, u)
    C.seed_everything(u['seed'])
    torch.backends.cudnn.benchmark = True
    parts = prots[u['split']]
    tr = windows(parts['train'], seq, TRAIN_WIN_PER_SEG)
    va = windows(parts['val'], seq, EVAL_WIN_PER_SEG)
    te = windows(parts['test'], seq, EVAL_WIN_PER_SEG)
    m = u['model']
    CF = maa.CFG
    cfg = dict(epochs=40, lr=CF['LR'][m], wd=CF['WD'], batch=CF['BATCH'][m], accum=CF['ACCUM'][m],
               patience=8, workers=CF['WORKERS'], warmup=CF['WARMUP'], amp=True)
    C.log(f'windows train {len(tr)} / val {len(va)} / test {len(te)}')
    t0 = time.time()
    C.train_classifier(rdir, build_k(m, K), maa.SkelDS(tr, seq, F, True, u['aug']),
                       maa.SkelDS(va, seq, F, False, u['aug']), cfg)
    res = maa.evaluate(rdir, build_k(m, K), maa.SkelDS(te, seq, F, False, u['aug']), F, cfg)
    res.update(u, minutes=round((time.time() - t0) / 60, 1), gpu=torch.cuda.get_device_name(0))
    C.save_json(res, f'{rdir}/DONE.json')
    C.log(f"{u} -> wheelchair {res.get('vid_f1_dis', float('nan')):.4f}, non-wheelchair {res.get('vid_f1_nondis', float('nan')):.4f}")


def execute(P, seq, F, prots, K, todo):
    rnd = random.Random(C.SESSION)
    todo = list(todo)
    rnd.shuffle(todo)
    for u in todo:
        rdir = unit_dir(P, u)
        if os.path.exists(f'{rdir}/DONE.json') or not C.claim_run(rdir):
            continue
        if maa._done_elsewhere(rdir):
            C.release_run(rdir)
            continue
        C.log(f'===== {u} =====')
        try:
            run_unit(P, seq, F, prots, K, u)
        finally:
            C.release_run(rdir)
    left = [u for u in todo if not os.path.exists(f'{unit_dir(P, u)}/DONE.json')]
    C.log(f'Session finished. Open units: {len(left)}')


def summary(P, exp='S9', K=None, n_boot=2000, base='A0'):
    """Segment-level macro-F1 (%) mean +- sd over seeds, and paired bootstrap difference vs A0 of the same model."""
    import pandas as pd
    rows = [json.load(open(f)) for f in glob.glob(f'{P.work}/maa/runs/{exp}/*/*/*/*/DONE.json')]
    if not rows:
        return None, None
    df = pd.DataFrame(rows)
    out = []
    for (aug, model, prot), g in df.groupby(['aug', 'model', 'split']):
        r = dict(protocol=prot, model=model, aug=aug, seeds=len(g))
        for col, name in (('vid_f1_dis', 'wheelchair'), ('vid_f1_nondis', 'non-wheelchair')):
            if col in g and g[col].notna().any():
                v = g[col].dropna() * 100
                r[name] = f'{v.mean():.1f} ± {v.std(ddof=1) if len(v) > 1 else 0:.1f}'
        out.append(r)
    s = pd.DataFrame(out).sort_values(['protocol', 'model', 'aug'])
    # paired bootstrap on the wheelchair test segments, vs A0
    rng = np.random.default_rng(0)
    comp = []
    for (model, prot), g in df.groupby(['model', 'split']):
        for aug in sorted(set(g['aug']) - {base}):
            pairs = []
            for sd in sorted(set(g['seed'])):
                fb = f"{P.work}/maa/runs/{exp}/{base}/{model}/{prot}/s{sd}/video_probs.npz"
                fv = f"{P.work}/maa/runs/{exp}/{aug}/{model}/{prot}/s{sd}/video_probs.npz"
                if os.path.exists(fb) and os.path.exists(fv):
                    pairs.append((np.load(fb, allow_pickle=True), np.load(fv, allow_pickle=True)))
            if not pairs:
                continue
            for grp in ('dis', 'nondis'):
                idx = np.where(pairs[0][0]['group'] == grp)[0]
                if len(idx) == 0:
                    continue
                k = int(K or pairs[0][0]['probs'].shape[1])
                Y = np.stack([b['y'][idx] for b, _ in pairs])
                PB = np.stack([b['probs'][idx].argmax(1) for b, _ in pairs])
                PV = np.stack([v['probs'][idx].argmax(1) for _, v in pairs])
                f = lambda y, p: maa._f1(y, p, k)
                d0 = np.mean([f(Y[i], PV[i]) - f(Y[i], PB[i]) for i in range(len(pairs))])
                bs = []
                for _ in range(n_boot):
                    r_ = rng.integers(0, len(idx), len(idx))
                    bs.append(np.mean([f(Y[i, r_], PV[i, r_]) - f(Y[i, r_], PB[i, r_]) for i in range(len(pairs))]))
                lo, hi = np.percentile(bs, [2.5, 97.5]) * 100
                comp.append(dict(protocol=prot, model=model, aug=aug, group='wheelchair' if grp == 'dis' else 'non-wheelchair',
                                 seeds=len(pairs), segments=len(idx), diff=round(d0 * 100, 1), ci_lo=round(lo, 1),
                                 ci_hi=round(hi, 1), significant=bool(lo > 0 or hi < 0)))
    c = pd.DataFrame(comp)
    os.makedirs(f'{P.work}/maa/results', exist_ok=True)
    s.to_csv(f'{P.work}/maa/results/S9_summary.csv', index=False)
    c.to_csv(f'{P.work}/maa/results/S9_compare_vs_{base}.csv', index=False)
    return s, c


In [ ]:
import sys, importlib
sys.path.insert(0, '/content')
import augs, maa, maa_safer as S
for m in (augs, maa, S): importlib.reload(m)
from carr import common as C
SAFER = next(c for c in [f'{DATA}/SAFER', '/content/drive/MyDrive/SAFER'] if os.path.isdir(c))
P = maa.paths(DATA, DATA)
obj = S.load_or_convert(SAFER, f'{DATA}/maa/safer_converted.pkl')
seq, F, PROTS, K, INFO = S.prepare(obj)
S._patch_metrics(K)
print('Kelas:', INFO['classes'])
print('Jumlah segmen tes kursi roda per kelas:', {c: obj['wc_test_counts'].get(c) for c in INFO['classes']})
print({p: {k: len(v) for k, v in d.items()} for p, d in PROTS.items()})
print({k: v for k, v in INFO.items() if k != 'classes'})

In [ ]:
TODO = [dict(exp='S9', aug=a, model=m, split='SZ', seed=s) for a in AUGS_SZ for m in MODELS for s in SEEDS] + \
       [dict(exp='S9', aug=a, model=m, split='SO', seed=s) for a in AUGS_SO for m in MODELS for s in SEEDS]
print(len(TODO), 'unit')
S.execute(P, seq, F, PROTS, K, TODO)

## Ringkasan dan uji beda terhadap A0 (bisa dijalankan kapan saja)

In [ ]:
import pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_rows', 300)
s, c0 = S.summary(P, K=K)
_, c4 = S.summary(P, K=K, base='A4')
display(s); print('=== vs A0 ==='); display(c0); print('=== vs A4 ==='); display(c4)
print('Disimpan di', f'{DATA}/maa/results/S9_summary.csv, S9_compare_vs_A0.csv, S9_compare_vs_A4.csv')
